# 33. Fine-Tuning Readiness and Minimal Intervention | 微调准入与最小干预
**难度：** Medium | **环境：** CPU-only | **标签：** `后训练`, `准入决策`, `评测` | **目标人群：** 微调项目设计者

---

## 本节导读

微调不是所有问题的默认答案。上线前应先确认失败是否稳定复现，再比较提示词、检索增强生成（RAG）和监督微调（SFT）能否解决问题，并检查数据、评测、资源和交付条件。本节实现的是最小可行干预选择，而不是训练参数报表。

**关键词：** `readiness gate`, `minimal intervention`, `Prompt`, `RAG`, `SFT`

---

## 前置阅读

**导语：** 进入本节前，先理解可比较实验为什么需要固定目标、工作负载与基线。
- [20. Performance Evidence and Experiment Design | 性能证据与实验设计](../00_Prerequisites/20_Performance_Evidence_and_Experiment_Design.ipynb)

---


### Step 1：用失败证据确定是否进入干预链

先固定目标、门槛、工作负载和失败切片，再判断问题是否稳定存在。基线必须覆盖全部失败切片，独立评测必须与当前候选训练隔离；否则此时只能补证据，不能选择训练方法。

| 证据 | 准入问题 | 缺失时的动作 |
|:---|:---|:---|
| 可测目标 | 指标和验收门槛是否明确 | 先定义目标与门槛 |
| 重复失败 | 失败是否在多次运行中稳定出现 | 固定工作负载并复现 |
| 失败切片 | 哪些场景、任务或样本持续失败 | 建立切片级基线 |
| 独立评测 | 是否能检验未见数据上的泛化 | 建立隔离评测集 |

![微调准入决策链](./../docs/public/02_PyTorch_Algorithms/33_finetuning_readiness_flow.svg)

### Step 2：沿知识、行为与反馈三类根因选择候选

干预方法先按根因分流。表达问题优先修改提示，外部知识优先使用检索增强生成（RAG），模型内部的领域表示缺口才考虑持续预训练，稳定的任务接口与行为缺口进入监督微调（SFT）。SFT 基线已经可用后，成对偏好适合离线偏好优化，需要探索或可验证结果时才进入在线强化学习。

| 候选 | 能覆盖的失败信号 | 进入执行前的必要条件 |
|:---|:---|:---|
| 提示模板 | 指令歧义、格式不稳定 | 固定模板后可复测 |
| RAG | 新知识、私有知识、可追溯来源 | 可用语料、检索评测与引用链 |
| 持续预训练 | 领域语言或内部知识表示不足 | 领域语料、通用能力回归集和训练预算 |
| SFT | 重复行为缺口、风格或任务接口 | 高质量示范、独立评测和训练预算 |
| 离线偏好优化 | 多个可行回答之间的偏好或安全选择 | SFT 基线、偏好对和独立评测 |
| 在线 RL / RLVR | 需要探索新轨迹或结果可由规则验证 | 奖励 / 验证器、轨迹预算和回滚条件 |

![从非参数方法到后训练分支的干预矩阵](./../docs/public/02_PyTorch_Algorithms/33_intervention_matrix.svg)

### Step 3：用完整覆盖与硬门槛选择最小充分干预

候选必须覆盖当前全部失败信号，而不是命中其中任意一个；随后再检查数据、评测、资源和交付条件。通过两道硬门槛后，才按显式干预顺序选择最小方案。该顺序表达方法边界，不使用任意权重把覆盖不足与低成本相互抵消。

| 决策量 | 检查内容 | 对选择的影响 |
|:---|:---|:---|
| 机制覆盖 | 候选是否覆盖全部当前失败信号 | 只覆盖一部分仍不得进入排序 |
| 执行条件 | 数据、语料、评测、资源和交付是否齐备 | 缺失则记录阻塞原因 |
| 干预顺序 | 提示、RAG、持续预训练、SFT、偏好优化、在线 RL | 选择首个完整覆盖且条件齐备的候选 |
| 阻塞证据 | 哪些必要条件仍缺失 | 没有就绪候选时返回具体缺口 |

### Step 4：实现最小干预准入决策

题目区先检查失败切片是否具备完整证据，再计算每个候选已覆盖和未覆盖的失败信号，最后按调用方给出的干预顺序选择首个完整覆盖且条件齐备的候选。规格与顺序由骨架提供，TODO 不要求拼接字符串或设计评分权重。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|:---|:---|:---|:---|
| TODO 1：检查问题证据<br>`assess_problem_evidence` | 找出缺失基础证据和未覆盖失败切片 | 两类缺口分别返回 | `test_problem_evidence_gate` |
| TODO 2：建立候选状态<br>`build_intervention_candidates` | 计算已覆盖、未覆盖信号与缺失条件 | 只有完整覆盖才算机制匹配 | `test_intervention_candidate_coverage` |
| TODO 3：选择最小干预<br>`recommend_minimal_intervention` | 按显式顺序筛选完整覆盖且就绪的候选 | 不使用综合分数；保留阻塞原因 | `test_minimal_intervention_decision`、`test_no_ready_candidate` |

## 测试

运行下方 CPU 机制测试，检查问题证据、候选准入和最小干预决策。


In [ ]:
from typing import Dict, List


In [ ]:
def assess_problem_evidence(evidence: Dict[str, object]) -> Dict[str, object]:
    """检查基础证据，并确认基线覆盖全部稳定失败切片。"""
    required = ["goal_metric", "target_threshold", "repeatable_failure", "independent_evaluation"]
    failure_slices = set(evidence.get("failure_slices", []))
    baseline_slices = set(evidence.get("baseline_slices", []))
    # ==========================================
    # TODO 1：分别计算缺失基础证据和未被基线覆盖的失败切片。
    # missing_foundations = ???
    # uncovered_failure_slices = ???
    # ready = ???  # 还要求至少存在一个稳定失败切片。
    # ==========================================
    raise NotImplementedError("TODO 1：检查问题证据")
    return {"ready": ready, "missing_foundations": missing_foundations, "uncovered_failure_slices": uncovered_failure_slices}


def build_intervention_candidates(context: Dict[str, object], specifications: List[Dict[str, object]]) -> List[Dict[str, object]]:
    """按外部规格建立候选的机制覆盖与执行条件证据。"""
    failure_signals = set(context.get("failure_signals", []))
    candidates = []
    for spec in specifications:
        # ==========================================
        # TODO 2：分别计算已覆盖、未覆盖的失败信号，以及尚未满足的必要条件。
        # covered_signals = ???  # 当前候选能够解释的失败信号。
        # uncovered_signals = ???  # 当前候选仍无法解释的失败信号。
        # missing_requirements = ???  # 执行当前候选仍缺少的硬条件。
        # ==========================================
        raise NotImplementedError("TODO 2：建立干预候选状态")
        candidates.append({"name": spec["name"], "covered_signals": covered_signals, "uncovered_signals": uncovered_signals, "mechanism_match": bool(failure_signals) and not uncovered_signals, "requirements_ready": not missing_requirements, "missing_requirements": missing_requirements})
    return candidates


def recommend_minimal_intervention(
    evidence_report: Dict[str, object],
    candidates: List[Dict[str, object]],
    intervention_order: List[str],
) -> Dict[str, object]:
    """在机制匹配且条件就绪的候选中选择最小充分干预。"""
    if not evidence_report["ready"]:
        return {"decision": "blocked", "reason": "missing_problem_evidence", "missing": evidence_report["missing_foundations"] + evidence_report["uncovered_failure_slices"]}
    # ==========================================
    candidate_by_name = {candidate["name"]: candidate for candidate in candidates}
    # TODO 3：按显式干预顺序筛选完整覆盖且执行条件齐备的候选。
    # ready_candidates = ???  # 顺序必须来自 intervention_order。
    # ==========================================
    raise NotImplementedError("TODO 3：选择最小可行干预")
    if ready_candidates:
        return {"decision": ready_candidates[0]["name"], "reason": "minimal_ready_intervention", "missing": []}
    missing = [item for candidate in candidates if candidate["mechanism_match"] for item in candidate["missing_requirements"]]
    return {"decision": "blocked", "reason": "candidate_requirements_not_ready", "missing": missing}

In [ ]:
def _intervention_specs():
    """候选规格由调用方提供，便于扩展而不改决策函数。"""
    return [
        {"name": "prompt", "failure_signals": {"format", "instruction"}, "requirements": ["prompt_tested"]},
        {"name": "rag", "failure_signals": {"fresh_knowledge", "private_knowledge"}, "requirements": ["corpus_ready", "retrieval_eval_ready"]},
        {"name": "continued_pretraining", "failure_signals": {"domain_language", "domain_knowledge"}, "requirements": ["domain_corpus_ready", "regression_eval_ready", "training_budget_ready"]},
        {"name": "sft", "failure_signals": {"behavior", "style", "task_pattern"}, "requirements": ["sft_data_ready", "training_budget_ready"]},
        {"name": "preference_optimization", "failure_signals": {"preference", "safety_choice"}, "requirements": ["sft_baseline_ready", "preference_data_ready"]},
        {"name": "online_rl", "failure_signals": {"exploration", "verifiable_outcome"}, "requirements": ["reward_or_verifier_ready", "rollout_budget_ready", "rollback_ready"]},
    ]


INTERVENTION_ORDER = ["prompt", "rag", "continued_pretraining", "sft", "preference_optimization", "online_rl"]

def test_problem_evidence_gate():
    """问题定义不完整时不能进入候选比较。"""
    report = assess_problem_evidence({"goal_metric": "accuracy", "target_threshold": 0.9, "repeatable_failure": True, "independent_evaluation": False, "failure_slices": ["json_format", "long_input"], "baseline_slices": ["json_format"]})
    assert report == {"ready": False, "missing_foundations": ["independent_evaluation"], "uncovered_failure_slices": ["long_input"]}


def test_intervention_candidate_coverage():
    """候选必须覆盖全部失败信号，部分命中不能冒充充分干预。"""
    specs = _intervention_specs()
    prompt = build_intervention_candidates({"failure_signals": ["format"], "prompt_tested": True}, specs)
    assert prompt[0]["mechanism_match"] is True and prompt[0]["requirements_ready"] is True
    rag = build_intervention_candidates({"failure_signals": ["fresh_knowledge"], "corpus_ready": True, "retrieval_eval_ready": True}, specs)
    assert rag[1]["mechanism_match"] is True and rag[1]["requirements_ready"] is True
    mixed = build_intervention_candidates({"failure_signals": ["format", "fresh_knowledge"], "prompt_tested": True}, specs)
    assert mixed[0]["mechanism_match"] is False and mixed[0]["uncovered_signals"] == ["fresh_knowledge"]
    online = build_intervention_candidates({"failure_signals": ["verifiable_outcome"], "reward_or_verifier_ready": True, "rollout_budget_ready": True, "rollback_ready": True}, specs)
    assert online[-1]["mechanism_match"] is True and online[-1]["requirements_ready"] is True


def test_minimal_intervention_decision():
    """决策只选择机制匹配且满足条件的最小候选。"""
    evidence = {"ready": True, "missing_foundations": [], "uncovered_failure_slices": []}
    candidates = build_intervention_candidates({"failure_signals": ["fresh_knowledge"], "corpus_ready": True, "retrieval_eval_ready": True, "sft_data_ready": True, "training_budget_ready": True}, _intervention_specs())
    assert recommend_minimal_intervention(evidence, candidates, INTERVENTION_ORDER)["decision"] == "rag"
    assert recommend_minimal_intervention({"ready": False, "missing_foundations": ["independent_evaluation"], "uncovered_failure_slices": []}, candidates, INTERVENTION_ORDER)["decision"] == "blocked"


def test_no_ready_candidate():
    """根因匹配但执行条件不足时，应返回具体阻塞字段。"""
    evidence = {"ready": True, "missing_foundations": [], "uncovered_failure_slices": []}
    candidates = build_intervention_candidates({"failure_signals": ["behavior"], "sft_data_ready": False, "training_budget_ready": False}, _intervention_specs())
    result = recommend_minimal_intervention(evidence, candidates, INTERVENTION_ORDER)
    assert result == {"decision": "blocked", "reason": "candidate_requirements_not_ready", "missing": ["sft_data_ready", "training_budget_ready"]}


try:
    test_problem_evidence_gate()
    test_intervention_candidate_coverage()
    test_minimal_intervention_decision()
    test_no_ready_candidate()
    print("✅ 微调问题证据、候选 gate 与最小干预测试通过。")
except (AttributeError, NameError, TypeError, ValueError, AssertionError) as error:
    raise NotImplementedError("请先完成 TODO 代码！") from error

---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def assess_problem_evidence(evidence: Dict[str, object]) -> Dict[str, object]:
    required = ["goal_metric", "target_threshold", "repeatable_failure", "independent_evaluation"]
    failure_slices = set(evidence.get("failure_slices", []))
    baseline_slices = set(evidence.get("baseline_slices", []))
    # TODO 1：检查基础证据与失败切片覆盖。
    missing_foundations = [name for name in required if not evidence.get(name)]
    uncovered_failure_slices = sorted(failure_slices - baseline_slices)
    ready = bool(failure_slices) and not missing_foundations and not uncovered_failure_slices
    return {"ready": ready, "missing_foundations": missing_foundations, "uncovered_failure_slices": uncovered_failure_slices}


def build_intervention_candidates(context: Dict[str, object], specifications: List[Dict[str, object]]) -> List[Dict[str, object]]:
    failure_signals = set(context.get("failure_signals", []))
    candidates = []
    for spec in specifications:
        # TODO 2：计算完整机制覆盖与缺失执行条件。
        supported_signals = set(spec["failure_signals"])
        covered_signals = sorted(failure_signals & supported_signals)
        uncovered_signals = sorted(failure_signals - supported_signals)
        missing_requirements = [item for item in spec["requirements"] if not context.get(item)]
        candidates.append({
            "name": spec["name"],
            "covered_signals": covered_signals,
            "uncovered_signals": uncovered_signals,
            "mechanism_match": bool(failure_signals) and not uncovered_signals,
            "requirements_ready": not missing_requirements,
            "missing_requirements": missing_requirements,
        })
    return candidates


def recommend_minimal_intervention(evidence_report: Dict[str, object], candidates: List[Dict[str, object]], intervention_order: List[str]) -> Dict[str, object]:
    if not evidence_report["ready"]:
        return {"decision": "blocked", "reason": "missing_problem_evidence", "missing": evidence_report["missing_foundations"] + evidence_report["uncovered_failure_slices"]}
    candidate_by_name = {candidate["name"]: candidate for candidate in candidates}
    # TODO 3：按外部给出的干预顺序选择首个完整覆盖且就绪的候选。
    ready_candidates = [candidate_by_name[name] for name in intervention_order if name in candidate_by_name and candidate_by_name[name]["mechanism_match"] and candidate_by_name[name]["requirements_ready"]]
    if ready_candidates:
        return {"decision": ready_candidates[0]["name"], "reason": "minimal_ready_intervention", "missing": []}
    missing = [item for candidate in candidates if candidate["mechanism_match"] for item in candidate["missing_requirements"]]
    return {"decision": "blocked", "reason": "candidate_requirements_not_ready", "missing": missing}

### 解析

#### TODO 1：检查问题证据

- 答案分别列出缺失的目标、阈值、稳定复现和独立评测证据，再计算未被基线覆盖的失败切片。
- 只有存在稳定失败切片且两类缺口都为空时才允许选方案；测试分别验证基础证据不足和切片覆盖不足。

#### TODO 2：建立候选覆盖与准入条件

- 答案分别计算候选已覆盖和未覆盖的失败信号，并逐项检查数据、评测、资源或回滚条件。
- 只有未覆盖集合为空时才算机制匹配；测试专门验证部分命中不能冒充充分干预。

#### TODO 3：选择最小可行干预

- 答案按照调用方给出的干预顺序，选择首个完整覆盖且执行条件就绪的候选，不使用综合分数。
- 问题证据不足时直接 `blocked`；没有就绪候选时返回具体缺失条件，不为了得到结果而强行选择训练方法。

## 相关阅读

完成问题与干预准入后，可以继续检查 SFT 数据、训练循环与项目交付。

- [32. SFT Data Admission and Length Audit | SFT 数据准入与长度审计](./32_Data_Engineering_for_SFT.ipynb)
- [13. SFT Training Loop | SFT 训练循环](./13_SFT_Training_Loop.ipynb)
- [62. Instruction Fine-Tuning Project | 指令微调项目](./62_Instruction_Fine_Tuning_Project.ipynb)
